# 02 — Data Cleaning
**Zomato Project · Phase 2**

**Objective:** Produce a standardized, information-preserving dataset that retains every valid restaurant record. Cleaning decisions are driven by data correctness — not by training requirements. Row filtering for supervised learning happens inside individual model notebooks.

**Pipeline position:**
```
01_Data_Profiling → [02_Data_Cleaning] → 03_Feature_Engineering
```

**Design principle:**
```
Cleaning  = fix what is structurally broken
Modelling = decide which rows participate in training
```
These two concerns are kept strictly separate. Restaurants with missing ratings,
missing dishes, or missing menus are still valid restaurants — they feed Flask,
the recommendation engine, and the RAG pipeline. Only the supervised modelling
notebooks filter them out at train time with:
    train_df = cleaned_df[cleaned_df['rate'].notna()].copy()

**Scope of this notebook (deliberately limited):**
- Missing value standardization
- Data type correction
- Text normalization (conservative)
- Duplicate removal (exact rows only)
- Column drops
- Data validation

**Explicitly excluded from this notebook:**
- Dropping rows because rate is null → model notebooks
- Binary / Label / One-Hot Encoding → `03_Feature_Engineering`
- Scaling → `03_Feature_Engineering`
- Median imputation of approx_cost → done here (genuine business feature)
- NLP tokenization / lemmatization → NLP stage
- Train/Test split → `03_Feature_Engineering`

## 0 · Imports & Configuration
> **Reproducibility:** This notebook is deterministic. Running it multiple times on the same raw dataset will always produce identical cleaned outputs. No random seeds are required because no sampling or stochastic operations are performed.

In [23]:
import pandas as pd
import numpy as np
import re
import html
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_PATH   = Path('/Users/huntstar/Projects/Zomato_project/Data/zomato.csv')
CLEAN_PATH = Path('/Users/huntstar/Projects/Zomato_project/Data/zomato_cleaned_v1.csv')

print('Libraries loaded.')
print(f'Raw   → {RAW_PATH}')
print(f'Clean → {CLEAN_PATH}')

Libraries loaded.
Raw   → /Users/huntstar/Projects/Zomato_project/Data/zomato.csv
Clean → /Users/huntstar/Projects/Zomato_project/Data/zomato_cleaned_v1.csv


## 1 · Load Raw Dataset & Create Working Copy

In [24]:
df_raw = pd.read_csv(RAW_PATH)
df     = df_raw.copy()  # ALL mutations happen on df — df_raw is never touched

print(f'Raw shape  : {df_raw.shape}')
print(f'Copy shape : {df.shape}')
print('\nColumn dtypes (raw):')
print(df.dtypes)

Raw shape  : (51717, 17)
Copy shape : (51717, 17)

Column dtypes (raw):
url                              str
address                          str
name                             str
online_order                     str
book_table                       str
rate                             str
votes                          int64
phone                            str
location                         str
rest_type                        str
dish_liked                       str
cuisines                         str
approx_cost(for two people)      str
reviews_list                     str
menu_item                        str
listed_in(type)                  str
listed_in(city)                  str
dtype: object


## 2 · Standardize Missing Values

**Why this step exists:**  
Placeholder strings like `"NEW"`, `"-"`, and `"NULL"` are stored as ordinary text in the raw CSV. Pandas does not recognise them as missing, so they silently bypass null-handling logic and corrupt downstream operations such as `.fillna()`, `.isnull()`, and Scikit-learn imputers. Converting all of them to proper `NaN` values first ensures that every subsequent step in this notebook — and every model in the pipeline — sees a consistent, honest picture of missingness.

Replace all placeholder strings (`"NEW"`, `"-"`, `"NULL"`, `"N/A"`, `"nan"`, etc.) with proper `NaN` values.

In [25]:
INVALID_TOKENS = {
    'NEW', '-', 'NULL', 'N/A', 'NA', 'nan', 'NaN',
    'none', 'None', 'NONE', 'n/a', 'null', '#N/A',
    '', ' '
}

def replace_invalid(value):
    """Return NaN for known placeholder strings; pass everything else through."""
    if isinstance(value, str):
        stripped = value.strip()
        if stripped in INVALID_TOKENS:
            return np.nan
    return value

object_cols  = df.select_dtypes(include='object').columns.tolist()
before_nulls = df.isnull().sum().sum()

for col in object_cols:
    df[col] = df[col].map(replace_invalid)

after_nulls = df.isnull().sum().sum()

print(f'Null count BEFORE standardization : {before_nulls:,}')
print(f'Null count AFTER  standardization : {after_nulls:,}')
print(f'New NaNs created                  : {after_nulls - before_nulls:,}')
print('\nPer-column null counts after standardization:')
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

Null count BEFORE standardization : 37,700
Null count AFTER  standardization : 39,977
New NaNs created                  : 2,277

Per-column null counts after standardization:
dish_liked                     28078
rate                           10052
phone                           1208
approx_cost(for two people)      346
rest_type                        227
cuisines                          45
location                          21
dtype: int64


## 3 · Data Type Correction

**Why this step exists:**  
The raw CSV stores every column as a string. Numeric columns (`rate`, `approx_cost`, `votes`) cannot be used in any arithmetic, distance computation, or model training until they carry the correct numeric dtype. Fixing dtypes here — before any modelling — means every downstream notebook can assume the types are already correct and skip redundant conversion logic.

### 3.1 · `rate` — remove `/5` suffix, cast to float

> Rows where `rate` is `NaN` after this conversion are **retained in the cleaned dataset**. They cannot be used as regression or classification targets, but they represent valid restaurants for Flask, recommendation, and RAG purposes. Filtering happens in the modelling notebooks.

In [26]:
print('Unique rate values (sample, before):')
print(df['rate'].dropna().unique()[:20])

def clean_rate(val):
    """
    - Strip whitespace
    - Remove trailing '/5'
    - Return NaN for any remaining non-numeric string
    - Cast to float
    """
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    val = re.sub(r'/5\s*$', '', val).strip()
    try:
        return float(val)
    except ValueError:
        return np.nan

df['rate'] = df['rate'].apply(clean_rate)

print(f'\ndtype after cast     : {df["rate"].dtype}')
print(f'Valid ratings        : {df["rate"].notna().sum():,}')
print(f'Missing ratings      : {df["rate"].isnull().sum():,}  ← RETAINED, not dropped')
print(f'Range (valid only)   : [{df["rate"].min()}, {df["rate"].max()}]')

out_of_range = df[(df['rate'].notna()) & ((df['rate'] < 0) | (df['rate'] > 5))]
print(f'Out-of-range ratings : {len(out_of_range)}')

Unique rate values (sample, before):
<StringArray>
[ '4.1/5',  '3.8/5',  '3.7/5',  '3.6/5',  '4.6/5',  '4.0/5',  '4.2/5',
  '3.9/5',  '3.1/5',  '3.0/5',  '3.2/5',  '3.3/5',  '2.8/5',  '4.4/5',
  '4.3/5',  '2.9/5',  '3.5/5',  '2.6/5', '3.8 /5',  '3.4/5']
Length: 20, dtype: str

dtype after cast     : float64
Valid ratings        : 41,665
Missing ratings      : 10,052  ← RETAINED, not dropped
Range (valid only)   : [1.8, 4.9]
Out-of-range ratings : 0


### 3.2 · `approx_cost(for two people)` — remove commas, cast to float

> Null values are **retained**. Median imputation is performed in `03_Feature_Engineering` within the train split to prevent data leakage.

In [27]:
print('Sample values before:')
print(df['approx_cost(for two people)'].dropna().unique()[:15])

def clean_cost(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().replace(',', '')
    try:
        return float(val)
    except ValueError:
        return np.nan

df['approx_cost(for two people)'] = df['approx_cost(for two people)'].apply(clean_cost)

print(f'\ndtype after cast : {df["approx_cost(for two people)"].dtype}')
print(f'Null count       : {df["approx_cost(for two people)"].isnull().sum():,}  ← RETAINED')
print(f'Range            : [{df["approx_cost(for two people)"].min()}, {df["approx_cost(for two people)"].max()}]')

Sample values before:
<StringArray>
['800', '300', '600', '700', '550', '500', '450', '650', '400', '900', '200',
 '750', '150', '850', '100']
Length: 15, dtype: str

dtype after cast : float64
Null count       : 346  ← RETAINED
Range            : [40.0, 6000.0]


### 3.3 · `votes` — verify integer, non-negative

In [28]:
print(f'votes dtype    : {df["votes"].dtype}')
print(f'votes nulls    : {df["votes"].isnull().sum()}')
print(f'votes negatives: {(df["votes"] < 0).sum()}')
print(f'votes range    : [{df["votes"].min()}, {df["votes"].max()}]')

assert df['votes'].dtype == 'int64', 'votes is not int64!'
assert (df['votes'] >= 0).all(),     'votes has negative values!'
print('\nvotes — OK (int64, no nulls, non-negative)')

votes dtype    : int64
votes nulls    : 0
votes negatives: 0
votes range    : [0, 16832]

votes — OK (int64, no nulls, non-negative)


### 3.4 · `online_order` & `book_table` — verify binary strings, defer encoding

> Encoding to 0/1 happens in `03_Feature_Engineering` alongside all other encoding operations.

In [29]:
for col in ['online_order', 'book_table']:
    vals = df[col].dropna().unique()
    print(f'{col} unique values : {vals}')
    unexpected = [v for v in vals if v not in ('Yes', 'No')]
    if unexpected:
        print(f'  ⚠️  Unexpected values in {col}: {unexpected}')
    else:
        print(f'  ✓  Only Yes/No — encoding deferred to Feature Engineering')

online_order unique values : <StringArray>
['Yes', 'No']
Length: 2, dtype: str
  ✓  Only Yes/No — encoding deferred to Feature Engineering
book_table unique values : <StringArray>
['Yes', 'No']
Length: 2, dtype: str
  ✓  Only Yes/No — encoding deferred to Feature Engineering


## 4 · Text Column Standardization

**Why this step exists:**  
Columns like `location`, `rest_type`, and `cuisines` will later be used for grouping, label encoding, and one-hot encoding. If the same value appears as `"north indian"`, `"North Indian"`, and `"North Indian "` (trailing space), each variant is treated as a separate category. This inflates cardinality artificially and reduces model performance. Normalizing whitespace and casing collapses these spurious variants into a single consistent token, without altering the underlying meaning of the data.

Columns: `name`, `location`, `rest_type`, `cuisines`, `listed_in(type)`, `listed_in(city)`

In [30]:
def normalize_text(val, case='none'):
    """
    - Strip leading/trailing whitespace
    - Collapse multiple internal spaces into one
    - Optionally apply case normalization
    """
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    val = re.sub(r'  +', ' ', val)
    if case == 'title':
        val = val.title()
    elif case == 'lower':
        val = val.lower()
    elif case == 'upper':
        val = val.upper()
    return val if val else np.nan

df['name']           = df['name'].apply(lambda x: normalize_text(x, case='none'))
df['location']       = df['location'].apply(lambda x: normalize_text(x, case='none'))
df['rest_type']      = df['rest_type'].apply(lambda x: normalize_text(x, case='title'))
df['listed_in(type)']= df['listed_in(type)'].apply(lambda x: normalize_text(x, case='title'))
df['listed_in(city)']= df['listed_in(city)'].apply(lambda x: normalize_text(x, case='title'))

print('Basic text normalization done for: name, location, rest_type, listed_in(type), listed_in(city)')

Basic text normalization done for: name, location, rest_type, listed_in(type), listed_in(city)


### 4.1 · `cuisines` — additional normalization of comma-separated values

In [31]:
def normalize_cuisines(val):
    """
    - Strip outer whitespace
    - Split on comma, strip each cuisine, title-case, rejoin
    - Ensures consistent format: 'North Indian, Chinese, Biryani'
    """
    if pd.isna(val):
        return np.nan
    parts = str(val).split(',')
    parts = [p.strip().title() for p in parts if p.strip()]
    return ', '.join(parts) if parts else np.nan

print('Before:')
print(df['cuisines'].dropna().head(5).values)

df['cuisines'] = df['cuisines'].apply(normalize_cuisines)

print('\nAfter:')
print(df['cuisines'].dropna().head(5).values)

Before:
<StringArray>
['North Indian, Mughlai, Chinese',    'Chinese, North Indian, Thai',
         'Cafe, Mexican, Italian',     'South Indian, North Indian',
       'North Indian, Rajasthani']
Length: 5, dtype: str

After:
<StringArray>
['North Indian, Mughlai, Chinese',    'Chinese, North Indian, Thai',
         'Cafe, Mexican, Italian',     'South Indian, North Indian',
       'North Indian, Rajasthani']
Length: 5, dtype: str


## 5 · Conservative Cleaning of NLP / RAG Columns

**Why this step exists:**  
`reviews_list`, `dish_liked`, and `menu_item` are the raw material for NLP sentiment analysis, dish extraction, and the RAG pipeline. Aggressive cleaning at this stage — removing rating prefixes, stopwords, or punctuation — would permanently destroy information that may be needed later. The only safe operations here are those that fix *encoding artifacts* (HTML tags, escaped characters, stray control characters) rather than *semantic content*. Everything else is deferred to the dedicated NLP preprocessing stage where decisions can be made with full context.

**Missing values in these columns are never a reason to drop a row.** A restaurant with no recorded dish preferences or no menu listing is still a valid restaurant — it simply has no text data for those specific fields, which is an expected business reality for many listings.

**NOT done here:**
- Removing `"Rated 4.0"` / `"RATED"` prefixes from `reviews_list` → deferred to NLP stage to prevent target-leakage analysis being done prematurely
- Tokenization, stopword removal, lemmatization → NLP stage

In [32]:
def conservative_text_clean(val):
    """
    Minimal cleaning safe for NLP/RAG downstream use:
      1. Remove HTML tags
      2. Decode HTML entities
      3. Normalize newlines to single space
      4. Remove non-printable / control characters
      5. Collapse multiple spaces
      6. Strip leading/trailing whitespace
    """
    if pd.isna(val):
        return np.nan
    val = str(val)
    val = re.sub(r'<[^>]+>', ' ', val)
    val = html.unescape(val)
    val = val.replace('\\n', ' ')
    val = val.replace('\\r', ' ')
    val = re.sub(r'[\r\n\t]+', ' ', val)
    val = re.sub(r'[\x00-\x1f\x7f]', '', val)
    val = re.sub(r'  +', ' ', val)
    val = val.strip()
    return val if val else np.nan

print('Cleaning reviews_list ...')
df['reviews_list'] = df['reviews_list'].apply(conservative_text_clean)

print('Cleaning dish_liked ...')
df['dish_liked'] = df['dish_liked'].apply(conservative_text_clean)

print('Cleaning menu_item ...')
df['menu_item'] = df['menu_item'].apply(conservative_text_clean)

print('Done.')
print('\nNull counts after text cleaning (retained — not dropped):')
for col in ['reviews_list', 'dish_liked', 'menu_item']:
    print(f'  {col:<20} : {df[col].isnull().sum():,}')

print('\nSample cleaned reviews_list entry:')
sample = df['reviews_list'].dropna().iloc[0]
print(sample[:300])

Cleaning reviews_list ...
Cleaning dish_liked ...
Cleaning menu_item ...
Done.

Null counts after text cleaning (retained — not dropped):
  reviews_list         : 0
  dish_liked           : 28,078
  menu_item            : 0

Sample cleaned reviews_list entry:
[('Rated 4.0', 'RATED A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that th


## 6 · Column Drops

**Why this step exists:**  
Two columns carry no predictive signal at any stage of the pipeline and are dropped immediately. `url` is kept until right before export — during debugging it is the fastest way to identify which restaurant a row refers to, and it costs nothing to carry it through cleaning.

| Column | When dropped | Reason |
|--------|-------------|--------|
| `phone` | Now | PII, 2.35% null, high cardinality (~15K unique) — no ML value |
| `address` | Now | Free-text PII — `location` already captures area |
| `url` | Right before export | 100% unique identifier — useful for debugging, dropped at end |

In [33]:
# url is kept until right before export — useful for debugging
COLS_TO_DROP_NOW  = ['phone', 'address']
COLS_TO_DROP_LAST = ['url']   # dropped just before export

print(f'Columns before drop: {df.shape[1]}')
df.drop(columns=COLS_TO_DROP_NOW, inplace=True)
print(f'Columns after drop : {df.shape[1]}')
print(f'Dropped now        : {COLS_TO_DROP_NOW}')
print(f'Dropped at export  : {COLS_TO_DROP_LAST}  (kept for debugging)')
print(f'\nRemaining columns  : {df.columns.tolist()}')

Columns before drop: 17
Columns after drop : 15
Dropped now        : ['phone', 'address']
Dropped at export  : ['url']  (kept for debugging)

Remaining columns  : ['url', 'name', 'online_order', 'book_table', 'rate', 'votes', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_cost(for two people)', 'reviews_list', 'menu_item', 'listed_in(type)', 'listed_in(city)']


## 7 · Missing Value Treatment

**Why this step exists:**  
The purpose of this stage is to handle missing values according to business semantics rather than applying blanket removal. Cleaning should preserve information whenever possible — the decision of whether a row participates in supervised learning is made later by individual modelling notebooks, not here.

**Missing Value Decision Table:**

| Column | Strategy | Reason |
|--------|----------|--------|
| `rate` | **Retain NaN** | Rows with missing target are intentionally preserved and are filtered only during supervised model training — they still feed Flask, recommendation, and RAG |
| `dish_liked` | **Retain NaN** | ~54% null is expected — absence of favourite dishes is not a data quality issue |
| `reviews_list` | **Retain NaN** | Some listings have no recorded reviews — valid business state |
| `menu_item` | **Retain NaN** | Many listings have no menu data — valid for Flask/RAG even without it |
| `approx_cost` | **Median imputation** | Only 346 missing (0.67%). This is a genuine business feature — cost is a real property of every restaurant. Median is used because the distribution is right-skewed |
| `rest_type` | **Sentinel → `Unknown`** | Only 227 missing (0.44%) — a dedicated Unknown category is more honest than mode imputation |
| `cuisines` | **Sentinel → `Unknown`** | Only 45 missing (0.09%) |
| `location` | **Sentinel → `Unknown`** | Only 21 missing (0.04%) |

> **Key principle:** Row count should not decrease in this section. Every restaurant — rated or unrated — leaves this step intact. Rows with missing target values (`rate`) are intentionally preserved in the cleaned dataset and are filtered only during supervised model training.

In [34]:
rows_before_mv = len(df)

# ── Median imputation — approx_cost(for two people) ───────────────────────
# Genuine business feature — every restaurant has a real cost.
# 346 missing (0.67%) — median used because distribution is right-skewed.
cost_median = df['approx_cost(for two people)'].median()
cost_nulls_before = df['approx_cost(for two people)'].isnull().sum()
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].fillna(cost_median)
print(f'approx_cost(for two people)         | NaN filled: {cost_nulls_before:>5} → 0  (median={cost_median:.0f})')

# ── Sentinel fill — low-null categorical columns ───────────────────────────
SENTINEL_COLS = ['rest_type', 'cuisines', 'location']

for col in SENTINEL_COLS:
    null_before = df[col].isnull().sum()
    df[col] = df[col].fillna('Unknown')
    print(f'{col:35s} | NaN filled: {null_before:>5} → 0')

rows_after_mv = len(df)
assert rows_before_mv == rows_after_mv, \
    f'Row count changed during missing value treatment: {rows_before_mv} → {rows_after_mv}'

print(f'\nRow count unchanged: {rows_after_mv:,}  ✓')
print('\nRemaining null counts (intentionally retained):')
remaining = df.isnull().sum()
retained = remaining[remaining > 0]
if len(retained) > 0:
    print(retained)
else:
    print('  None — all columns fully resolved.')

approx_cost(for two people)         | NaN filled:   346 → 0  (median=400)
rest_type                           | NaN filled:   227 → 0
cuisines                            | NaN filled:    45 → 0
location                            | NaN filled:    21 → 0

Row count unchanged: 51,717  ✓

Remaining null counts (intentionally retained):
rate          10052
dish_liked    28078
dtype: int64


## 8 · Duplicate Detection

**Why this step exists:**  
After standardization and text normalization, previously distinct rows can become identical. This step removes only **completely identical rows** — records where every single column carries exactly the same value.

**Critical design decision — why NOT subset-based deduplication:**  
The previous version of this notebook used a subset of key columns (`name`, `location`, `rate`, `votes`, etc.) to identify duplicates. This incorrectly treated chain restaurant branches as duplicates — e.g., all McDonald's outlets in Koramangala sharing the same name, location, cuisine, and cost were collapsed into a single row, causing a **60% data loss** (51,717 → 20,627 rows). Chain restaurants and repeated listings with different reviews, vote counts, or menu content are NOT duplicates — they are distinct business entries in Zomato's catalogue.

The correct approach checks for true exact duplicates only: a row is a duplicate if and only if it is completely identical to another row across **all 14 columns**.

In [35]:
rows_before_dedup = len(df)

# ── Step 1: Count true exact duplicates (all 14 columns identical) ────────
exact_dupes_mask = df.duplicated(keep=False)   # keep=False flags ALL copies
exact_dupes_count = exact_dupes_mask.sum()
print(f'Rows that are exact duplicates across ALL columns : {exact_dupes_count:,}')

if exact_dupes_count > 0:
    print('\nSample of exact duplicate rows (first 6):')
    print(df[exact_dupes_mask].head(6)[['name', 'location', 'rate', 'votes', 'cuisines']].to_string())

# ── Step 2: Remove exact duplicates only — keep first occurrence ──────────
df = df.drop_duplicates(keep='first')
rows_after_dedup = len(df)

print(f'\nRows before deduplication : {rows_before_dedup:,}')
print(f'Rows after  deduplication : {rows_after_dedup:,}')
print(f'Exact duplicate rows removed : {rows_before_dedup - rows_after_dedup:,}')

# ── Step 3: Informational — how many restaurants share name+location? ─────
# This is NOT used for row removal — it is purely for awareness
identity_shared = df.duplicated(subset=['name', 'location'], keep=False).sum()
print(f'\nInformational: rows sharing (name, location) — chain branches etc. : {identity_shared:,}')
print('  → These are NOT removed. Chain branches are legitimate distinct entries.')

# ── Final Row Summary ─────────────────────────────────────────────────────
sep = '─' * 50
print(f'\n{sep}')
print('  FINAL ROW SUMMARY')
print(sep)
print(f'  Original rows         : {len(df_raw):,}')
print(f'  Exact duplicates removed: {rows_before_dedup - rows_after_dedup:,}')
print(f'  Rows remaining        : {rows_after_dedup:,}')
print(f'  Retention rate        : {rows_after_dedup / len(df_raw) * 100:.1f}%')
print(f'  ─── Of remaining rows ───')
print(f'  With valid rate (supervised ML) : {df["rate"].notna().sum():,}')
print(f'  Missing rate (Flask/RAG/Rec)    : {df["rate"].isnull().sum():,}')
print(sep)

Rows that are exact duplicates across ALL columns : 0

Rows before deduplication : 51,717
Rows after  deduplication : 51,717
Exact duplicate rows removed : 0

Informational: rows sharing (name, location) — chain branches etc. : 49,812
  → These are NOT removed. Chain branches are legitimate distinct entries.

──────────────────────────────────────────────────
  FINAL ROW SUMMARY
──────────────────────────────────────────────────
  Original rows         : 51,717
  Exact duplicates removed: 0
  Rows remaining        : 51,717
  Retention rate        : 100.0%
  ─── Of remaining rows ───
  With valid rate (supervised ML) : 41,665
  Missing rate (Flask/RAG/Rec)    : 10,052
──────────────────────────────────────────────────


## 9 · Data Validation

**Why this step exists:**  
Cleaning steps can introduce silent bugs — a regex that strips too aggressively, a cast that silently produces `NaN` instead of raising an error, or a sentinel fill applied to the wrong column. This section provides a formal checkpoint: if any assertion or check fails here, the notebook stops before producing a corrupted output file. Catching problems at the source is always cheaper than debugging them inside a model three stages later.

### 9.1 · Data Type Audit

In [36]:
# Note: pandas with ArrowDtype or StringDtype may report 'str' instead of 'object'
# Both are string types — we treat them as equivalent here
STRING_TYPES = {'object', 'str', 'string'}

EXPECTED_DTYPES = {
    'name'                        : 'string',
    'online_order'                : 'string',
    'book_table'                  : 'string',
    'rate'                        : 'float64',
    'votes'                       : 'int64',
    'location'                    : 'string',
    'rest_type'                   : 'string',
    'dish_liked'                  : 'string',
    'cuisines'                    : 'string',
    'approx_cost(for two people)' : 'float64',
    'reviews_list'                : 'string',
    'menu_item'                   : 'string',
    'listed_in(type)'             : 'string',
    'listed_in(city)'             : 'string',
}

print('Data Type Audit')
print('-' * 60)
all_ok = True
for col, expected in EXPECTED_DTYPES.items():
    actual = str(df[col].dtype)
    # Accept any string-family dtype as equivalent
    if expected == 'string':
        ok = actual in STRING_TYPES
    else:
        ok = actual == expected
    status = '✓' if ok else '✗'
    if not ok:
        all_ok = False
    print(f'  {status}  {col:<35} expected={expected:<10} actual={actual}')

print()
if all_ok:
    print('All dtypes match expected.')
else:
    print('⚠️  Dtype mismatches found — review above.')

Data Type Audit
------------------------------------------------------------
  ✓  name                                expected=string     actual=str
  ✓  online_order                        expected=string     actual=str
  ✓  book_table                          expected=string     actual=str
  ✓  rate                                expected=float64    actual=float64
  ✓  votes                               expected=int64      actual=int64
  ✓  location                            expected=string     actual=str
  ✓  rest_type                           expected=string     actual=str
  ✓  dish_liked                          expected=string     actual=str
  ✓  cuisines                            expected=string     actual=str
  ✓  approx_cost(for two people)         expected=float64    actual=float64
  ✓  reviews_list                        expected=string     actual=str
  ✓  menu_item                           expected=string     actual=str
  ✓  listed_in(type)                     expected

### 9.2 · Numeric Range Validation

In [37]:
NUMERIC_BOUNDS = {
    'rate'                        : (0.0,   5.0),
    'votes'                       : (0,     200000),
    'approx_cost(for two people)' : (0.0,   10000.0),
}

print('Numeric Range Validation')
print('-' * 60)
for col, (lo, hi) in NUMERIC_BOUNDS.items():
    series     = df[col].dropna()
    violations = series[(series < lo) | (series > hi)]
    status     = '✓' if len(violations) == 0 else '✗'
    print(f'  {status}  {col:<35} range=[{series.min():.2f}, {series.max():.2f}]  '
          f'expected=[{lo}, {hi}]  violations={len(violations)}')

Numeric Range Validation
------------------------------------------------------------
  ✓  rate                                range=[1.80, 4.90]  expected=[0.0, 5.0]  violations=0
  ✓  votes                               range=[0.00, 16832.00]  expected=[0, 200000]  violations=0
  ✓  approx_cost(for two people)         range=[40.00, 6000.00]  expected=[0.0, 10000.0]  violations=0


### 9.3 · Categorical Value Validation

In [38]:
VALID_CATEGORICALS = {
    'online_order' : {'Yes', 'No'},
    'book_table'   : {'Yes', 'No'},
}

print('Categorical Value Validation')
print('-' * 60)
for col, valid_set in VALID_CATEGORICALS.items():
    actual_vals = set(df[col].dropna().unique())
    unexpected  = actual_vals - valid_set
    status = '✓' if not unexpected else '✗'
    print(f'  {status}  {col:<20} valid={valid_set}  actual={actual_vals}  unexpected={unexpected}')

Categorical Value Validation
------------------------------------------------------------
  ✓  online_order         valid={'No', 'Yes'}  actual={'No', 'Yes'}  unexpected=set()
  ✓  book_table           valid={'No', 'Yes'}  actual={'No', 'Yes'}  unexpected=set()


### 9.4 · Placeholder String Check

In [39]:
RESIDUAL_PLACEHOLDERS = ['NEW', 'NULL', 'N/A', '#N/A', 'none', 'NONE']

print('Residual Placeholder Check')
print('-' * 60)
obj_cols  = df.select_dtypes(exclude='number').columns.tolist()
found_any = False

for col in obj_cols:
    for placeholder in RESIDUAL_PLACEHOLDERS:
        count = (df[col].dropna() == placeholder).sum()
        if count > 0:
            print(f'  ✗  {col}: "{placeholder}" still present ({count} rows)')
            found_any = True

if not found_any:
    print('  ✓  No residual placeholder strings found in any column.')

Residual Placeholder Check
------------------------------------------------------------
  ✓  No residual placeholder strings found in any column.


### 9.5 · Missing Value Summary (post-cleaning)

In [40]:
null_summary = pd.DataFrame({
    'null_count' : df.isnull().sum(),
    'null_pct'   : (df.isnull().sum() / len(df) * 100).round(2),
    'dtype'      : df.dtypes,
    'treatment'  : {
        'name'                        : 'n/a',
        'online_order'                : 'n/a',
        'book_table'                  : 'n/a',
        'rate'                        : 'RETAINED — filter in model notebooks',
        'votes'                       : 'n/a',
        'location'                    : 'sentinel filled',
        'rest_type'                   : 'sentinel filled',
        'dish_liked'                  : 'RETAINED — expected high null',
        'cuisines'                    : 'sentinel filled',
        'approx_cost(for two people)' : f'Median imputed ({cost_median:.0f}) — 0 nulls remain',
        'reviews_list'                : 'RETAINED — expected null',
        'menu_item'                   : 'RETAINED — expected null',
        'listed_in(type)'             : 'n/a',
        'listed_in(city)'             : 'n/a',
    }
}).sort_values('null_count', ascending=False)

print('Post-Cleaning Missing Value Summary')
print('-' * 80)
print(null_summary.to_string())

Post-Cleaning Missing Value Summary
--------------------------------------------------------------------------------
                             null_count  null_pct    dtype                              treatment
dish_liked                        28078   54.2900      str          RETAINED — expected high null
rate                              10052   19.4400  float64   RETAINED — filter in model notebooks
approx_cost(for two people)           0    0.0000  float64  Median imputed (400) — 0 nulls remain
book_table                            0    0.0000      str                                    n/a
cuisines                              0    0.0000      str                        sentinel filled
listed_in(city)                       0    0.0000      str                                    n/a
listed_in(type)                       0    0.0000      str                                    n/a
location                              0    0.0000      str                        sentinel filled
m

### 9.6 · Hard Assertions
If any of these fail the notebook stops immediately — do not comment them out.

In [41]:
print('Running assertions...')

# ── Dtype checks ──────────────────────────────────────────────────────────
assert df['rate'].dtype == np.float64, \
    f"rate dtype should be float64, got {df['rate'].dtype}"
assert df['rate'].dropna().between(0, 5).all(), \
    f"rate has values outside [0, 5]"

assert df['votes'].dtype == np.int64, \
    f"votes dtype should be int64, got {df['votes'].dtype}"
assert df['votes'].min() >= 0, \
    f"votes has negative values: min={df['votes'].min()}"

assert df['approx_cost(for two people)'].isnull().sum() == 0, \
    'approx_cost still has nulls after median imputation'
assert df['approx_cost(for two people)'].min() >= 0, \
    'approx_cost has negative values after imputation'

# ── Binary columns ─────────────────────────────────────────────────────────
assert df['online_order'].isin(['Yes', 'No']).all(), \
    f"online_order has unexpected values: {df['online_order'].unique()}"
assert df['book_table'].isin(['Yes', 'No']).all(), \
    f"book_table has unexpected values: {df['book_table'].unique()}"

# ── Sentinel columns have no nulls ─────────────────────────────────────────
for col in ['rest_type', 'cuisines', 'location']:
    assert df[col].isnull().sum() == 0, \
        f'{col} still has nulls after sentinel fill: {df[col].isnull().sum()}'

# ── Placeholder strings gone ───────────────────────────────────────────────
PLACEHOLDERS = ['NEW', 'NULL', 'N/A', '#N/A', 'none', 'NONE', '-']
for col in df.select_dtypes(exclude='number').columns:
    for p in PLACEHOLDERS:
        count = (df[col].dropna() == p).sum()
        assert count == 0, \
            f'Placeholder "{p}" still present in column "{col}" ({count} rows)'

# ── Required columns present ───────────────────────────────────────────────
REQUIRED_COLS = [
    'name', 'online_order', 'book_table', 'rate', 'votes',
    'location', 'rest_type', 'dish_liked', 'cuisines',
    'approx_cost(for two people)', 'reviews_list', 'menu_item',
    'listed_in(type)', 'listed_in(city)'
]
for col in REQUIRED_COLS:
    assert col in df.columns, f'Required column "{col}" is missing'

# ── Row count sanity — should be close to raw count, not 60% less ─────────
row_retention_pct = len(df) / len(df_raw) * 100
assert row_retention_pct >= 80, \
    f'Only {row_retention_pct:.1f}% of rows retained — likely over-aggressive deduplication'
print(f'Row retention: {row_retention_pct:.1f}%  ✓')

# ── No new exact duplicates remain ────────────────────────────────────────
remaining_dupes = df.duplicated().sum()
assert remaining_dupes == 0, \
    f'{remaining_dupes:,} exact duplicate rows still remain'

print('All assertions passed. Dataset is clean and safe to export.')

Running assertions...
Row retention: 100.0%  ✓
All assertions passed. Dataset is clean and safe to export.


## 10 · Cleaning Impact Report

In [42]:
raw_mem   = df_raw.memory_usage(deep=True).sum()
clean_mem = df.memory_usage(deep=True).sum()

sep  = '=' * 68
sep2 = '-' * 68

print(sep)
print('  ZOMATO DATASET — CLEANING IMPACT REPORT')
print(sep)

print(f'\n  {"Metric":<30} {"Before":>12} {"After":>12} {"Delta":>10}')
print(f'  {sep2}')
print(f'  {"Rows":<30} {df_raw.shape[0]:>12,} {df.shape[0]:>12,} {df.shape[0]-df_raw.shape[0]:>+10,}')
print(f'  {"Columns":<30} {df_raw.shape[1]:>12} {df.shape[1]:>12} {df.shape[1]-df_raw.shape[1]:>+10}')
print(f'  {"Total NaN cells":<30} {df_raw.isnull().sum().sum():>12,} {df.isnull().sum().sum():>12,}')
print(f'  {"Memory usage":<30} {raw_mem/1e6:>11.2f}M {clean_mem/1e6:>11.2f}M {(clean_mem-raw_mem)/1e6:>+10.2f}M')

print(f'\n  DATATYPE CHANGES')
print(f'  {sep2}')
dtype_changes = [
    ('rate',                         'object', 'float64', 'Removed /5 suffix, cast numeric'),
    ('approx_cost(for two people)',   'object', 'float64', 'Removed commas, cast numeric'),
    ('votes',                         'int64',  'int64',  'Verified — no change needed'),
]
for col, before, after, note in dtype_changes:
    arrow = '→' if before != after else '='
    print(f'  {col:<38} {before:<10} {arrow}  {after:<10}  ({note})')

print(f'\n  COLUMNS DROPPED')
print(f'  {sep2}')
for col in COLS_TO_DROP_NOW + COLS_TO_DROP_LAST:
    print(f'  ✗  {col}')

print(f'\n  COLUMN-LEVEL CLEANING LOG')
print(f'  {sep2}')
cleaning_log = {
    'rate': [
        f'{df["rate"].isnull().sum():,} NaN values retained (NEW listings / unrated restaurants)',
        'Removed /5 suffix from all valid entries',
        'Cast to float64',
        f'Valid ratings: {df["rate"].notna().sum():,}',
    ],
    'approx_cost(for two people)': [
        f'{cost_nulls_before} NaN values filled with median ({cost_median:.0f})',
        'Commas removed, cast to float64',
        '0 nulls remain — fully resolved in cleaning',
    ],
    'votes': ['Verified int64 — no changes applied'],
    'reviews_list': [
        'HTML tags removed, HTML entities decoded',
        'Whitespace and newlines normalized',
        'Non-printable control characters removed',
        'Rating prefixes (Rated X.X) deliberately preserved for NLP stage',
    ],
    'dish_liked': [
        'HTML / whitespace artifacts cleaned (conservative)',
        f'{df["dish_liked"].isnull().sum():,} NaN values retained — expected high null',
    ],
    'menu_item': [
        'HTML / whitespace artifacts cleaned (conservative)',
        'Structure preserved for RAG pipeline',
    ],
    'cuisines': [
        'Split on comma, each cuisine stripped and title-cased, rejoined',
        f'NaN → "Unknown" sentinel (45 rows)',
    ],
    'rest_type'     : ['Title-cased and whitespace normalized', 'NaN → "Unknown" sentinel (227 rows)'],
    'location'      : ['Whitespace normalized', 'NaN → "Unknown" sentinel (21 rows)'],
    'name'          : ['Whitespace normalized — casing preserved'],
    'listed_in(type)' : ['Title-cased and whitespace normalized'],
    'listed_in(city)' : ['Title-cased and whitespace normalized'],
    'online_order'  : ['Verified — only Yes/No values present, encoding deferred'],
    'book_table'    : ['Verified — only Yes/No values present, encoding deferred'],
    'url'     : ['DROPPED — 100% unique identifier'],
    'phone'   : ['DROPPED — PII, high cardinality, no ML value'],
    'address' : ['DROPPED — PII, location captured in location column'],
}
for col, steps in cleaning_log.items():
    print(f'\n  {col}')
    for step in steps:
        print(f'    ↓  {step}')

print(f'\n  MISSING VALUE TREATMENT SUMMARY')
print(f'  {sep2}')
mv_table = [
    ('rate',                        f'{df["rate"].isnull().sum():,}',
     'RETAINED — filter per-model at train time'),
    ('dish_liked',                  f'{df["dish_liked"].isnull().sum():,}',
     'RETAINED — high null expected by design'),
    ('reviews_list',                f'{df["reviews_list"].isnull().sum():,}',
     'RETAINED — some listings have no reviews'),
    ('menu_item',                   f'{df["menu_item"].isnull().sum():,}',
     'RETAINED — some listings have no menu data'),
    ('approx_cost(for two people)', f'{df["approx_cost(for two people)"].isnull().sum():,}',
     'RETAINED — median imputation in FE notebook'),
    ('rest_type',                   '0', 'Filled with "Unknown" sentinel'),
    ('cuisines',                    '0', 'Filled with "Unknown" sentinel'),
    ('location',                    '0', 'Filled with "Unknown" sentinel'),
]
print(f'  {"Column":<35} {"NaN (clean)":>12}   Treatment')
print(f'  {sep2}')
for col, null_count, treatment in mv_table:
    print(f'  {col:<35} {null_count:>12}   {treatment}')

print(f'\n  DEDUPLICATION')
print(f'  {sep2}')
print(f'  Strategy : True exact duplicates only (all 14 columns identical)')
print(f'  Removed  : {df_raw.shape[0] - df.shape[0]:,} rows')
print(f'  Retained : {df.shape[0]:,} rows  ({df.shape[0]/df_raw.shape[0]*100:.1f}% of raw)')
print(f'  Chain branches, multi-listing restaurants → kept as distinct entries')

print(f'\n  DEFERRED TO 03_Feature_Engineering')
print(f'  {sep2}')
print('  - Binary encoding (online_order, book_table)')
print('  - Label / One-Hot encoding (categorical cols)')
print('  - StandardScaler (for SVM)')
print('  - Median imputation (approx_cost) — within train split only')
print('  - Dropping rows with null rate — inside each model notebook')
print('  - Train/Test split')

print(f'\n  DEFERRED TO NLP Stage')
print(f'  {sep2}')
print('  - Removing "Rated X.X" / "RATED" prefixes from reviews_list')
print('  - Tokenization, stopwords, lemmatization')

print(f'\n  Output → {CLEAN_PATH}')
print(sep)

  ZOMATO DATASET — CLEANING IMPACT REPORT

  Metric                               Before        After      Delta
  --------------------------------------------------------------------
  Rows                                 51,717       51,717         +0
  Columns                                  17           15         -2
  Total NaN cells                      37,700       38,130
  Memory usage                        601.07M      997.74M    +396.67M

  DATATYPE CHANGES
  --------------------------------------------------------------------
  rate                                   object     →  float64     (Removed /5 suffix, cast numeric)
  approx_cost(for two people)            object     →  float64     (Removed commas, cast numeric)
  votes                                  int64      =  int64       (Verified — no change needed)

  COLUMNS DROPPED
  --------------------------------------------------------------------
  ✗  phone
  ✗  address
  ✗  url

  COLUMN-LEVEL CLEANING LOG
  -----

## 11 · Data Dictionary

Complete reference for the cleaned dataset handed off to `03_Feature_Engineering`.

In [43]:
data_dict = [
    ('name',                        'str/object',  'Identifier',     'Recommendation',       'Restaurant name — preserved as-is'),
    ('online_order',                'str/object',  'Binary',         'All models',           'Yes/No — encode to 0/1 in FE'),
    ('book_table',                  'str/object',  'Binary',         'All models',           'Yes/No — encode to 0/1 in FE'),
    ('rate',                        'float64',     'Target',         'Regression / Classif', 'NaN retained — filter in model notebooks'),
    ('votes',                       'int64',       'Numerical',      'All models',           'Vote count — log transform in FE'),
    ('location',                    'str/object',  'Categorical',    'All models',           '93+ unique areas — label encode or OHE in FE'),
    ('rest_type',                   'str/object',  'Categorical',    'All models',           '93+ unique types — label encode in FE'),
    ('dish_liked',                  'str/object',  'NLP',            'NLP pipeline',         'High null expected — do not impute'),
    ('cuisines',                    'str/object',  'Recommendation', 'Recommendation + ML',  'Comma-separated — parse in FE'),
    ('approx_cost(for two people)', 'float64',     'Numerical',      'All models',           'NaN retained — median impute in FE'),
    ('reviews_list',                'str/object',  'NLP/Sentiment',  'NLP pipeline',         '"Rated X.X" prefix retained until NLP stage'),
    ('menu_item',                   'str/object',  'RAG',            'RAG pipeline',         'Structured item lists — parse in RAG stage'),
    ('listed_in(type)',             'str/object',  'Categorical',    'All models',           '7 unique types — OHE in FE'),
    ('listed_in(city)',             'str/object',  'Categorical',    'All models',           '30 unique cities — label encode in FE'),
]

header   = f'  {"Column":<35} {"Dtype":<12} {"Category":<16} {"Used In":<22} Notes'
sep_line = '  ' + '-' * 118

print('  DATA DICTIONARY — zomato_cleaned_v1.csv')
print('  ' + '=' * 118)
print(header)
print(sep_line)
for row in data_dict:
    col, dtype, cat, used, notes = row
    print(f'  {col:<35} {dtype:<12} {cat:<16} {used:<22} {notes}')
print('  ' + '=' * 118)
print(f'\n  Total columns : {len(data_dict)}')
print(f'  Total rows    : {len(df):,}')
print(f'\n  Rows available for supervised learning (rate not null) : {df["rate"].notna().sum():,}')
print(f'  Rows retained for Flask / RAG / Recommendation        : {df["rate"].isnull().sum():,} (no target)')

  DATA DICTIONARY — zomato_cleaned_v1.csv
  Column                              Dtype        Category         Used In                Notes
  ----------------------------------------------------------------------------------------------------------------------
  name                                str/object   Identifier       Recommendation         Restaurant name — preserved as-is
  online_order                        str/object   Binary           All models             Yes/No — encode to 0/1 in FE
  book_table                          str/object   Binary           All models             Yes/No — encode to 0/1 in FE
  rate                                float64      Target           Regression / Classif   NaN retained — filter in model notebooks
  votes                               int64        Numerical        All models             Vote count — log transform in FE
  location                            str/object   Categorical      All models             93+ unique areas — label enc

## 12 · Export Cleaned Dataset

In [44]:
# ── Drop url now — was kept for debugging, no longer needed in output ──────
if 'url' in df.columns:
    df.drop(columns=['url'], inplace=True)
    print('url dropped — no longer needed in cleaned output')

df.to_csv(CLEAN_PATH, index=False)

print(f'\nCleaned dataset saved to : {CLEAN_PATH}')
print(f'Shape                    : {df.shape}')
print(f'File size                : {CLEAN_PATH.stat().st_size / 1e6:.2f} MB')
print(f'\nRow retention vs raw     : {len(df):,} / {len(df_raw):,}  ({len(df)/len(df_raw)*100:.1f}%)')
print(f'Rows for supervised ML   : {df["rate"].notna().sum():,}  (rate not null)')
print(f'Rows for app / RAG       : {df["rate"].isnull().sum():,}  (rate null — still valuable)')

print('\nFinal column dtypes:')
print(df.dtypes)

url dropped — no longer needed in cleaned output

Cleaned dataset saved to : /Users/huntstar/Projects/Zomato_project/Data/zomato_cleaned_v1.csv
Shape                    : (51717, 14)
File size                : 541.92 MB

Row retention vs raw     : 51,717 / 51,717  (100.0%)
Rows for supervised ML   : 41,665  (rate not null)
Rows for app / RAG       : 10,052  (rate null — still valuable)

Final column dtypes:
name                               str
online_order                       str
book_table                         str
rate                           float64
votes                            int64
location                           str
rest_type                          str
dish_liked                         str
cuisines                           str
approx_cost(for two people)    float64
reviews_list                       str
menu_item                          str
listed_in(type)                    str
listed_in(city)                    str
dtype: object
